In [1]:
# ResNet18 with continuous random rotation (0-360) + flips + color jitter.
# Test on unrotated images only (single test set).

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

DATA_DIR = r"E:\Deep Learning Spring 26\dataset"
BATCH_SIZE = 32
EPOCHS_STAGE1 = 4
EPOCHS_STAGE2 = 2
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-5
HIDDEN1 = 256
HIDDEN2 = 64
NUM_WORKERS = 0
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
train_dir = os.path.join(DATA_DIR, "train")
df = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

Train : 176020
Val   : 22002
Test  : 22003


In [3]:
# continuous rotation + crop to remove black corners + resize back to 96
# max inscribed square in a rotated 96x96 image is ~68x68 (at 45 degrees)
train_tf = transforms.Compose([
    transforms.RandomRotation(degrees=(0, 360)),
    transforms.CenterCrop(68),
    transforms.Resize(96),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CancerDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, rotation=0):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.rotation = rotation
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row.id + ".tif")).convert("RGB")
        if self.rotation != 0:
            img = img.rotate(self.rotation)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row.label, dtype=torch.float32)

In [4]:
train_loader = DataLoader(CancerDataset(train_df, train_dir, train_tf, rotation=0),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(CancerDataset(val_df, train_dir, eval_tf, rotation=0),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# single unrotated test set
test_loader = DataLoader(CancerDataset(test_df, train_dir, eval_tf, rotation=0),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
model = models.resnet18(weights="IMAGENET1K_V1")
for name, param in model.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, HIDDEN1), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(HIDDEN1, HIDDEN2), nn.ReLU(),
    nn.Linear(HIDDEN2, 1))
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
total_epochs = EPOCHS_STAGE1 + EPOCHS_STAGE2
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_STAGE1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

In [6]:
def train_one_epoch():
    model.train()
    total_loss = 0
    preds_list, labels_list = [], []
    for imgs, labels in tqdm(train_loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        labels = labels.unsqueeze(1).to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds_list.extend(torch.sigmoid(out).squeeze().detach().cpu().numpy())
        labels_list.extend(labels.squeeze().cpu().numpy())
    avg_loss = total_loss / len(train_loader)
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return avg_loss, accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

def evaluate(loader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds_list.extend(torch.sigmoid(model(imgs)).squeeze().cpu().numpy())
            labels_list.extend(labels.numpy())
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

In [7]:
best_auc = 0.0
for epoch in range(1, total_epochs + 1):
    if epoch == EPOCHS_STAGE1 + 1:
        print("\n>> Stage 2: unfreezing layer4\n")
        for name, param in model.named_parameters():
            if "layer4" in name:
                param.requires_grad = True
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_STAGE2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

    loss, train_acc, train_f1, train_auc = train_one_epoch()
    scheduler.step()
    val_acc, val_f1, val_auc = evaluate(val_loader)

    lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch}/{total_epochs} (lr={lr:.2e})")
    print(f"  Train  =>  Loss: {loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}  AUC: {train_auc:.4f}")
    print(f"  Val    =>  Acc: {val_acc:.4f}  F1: {val_f1:.4f}  AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model_continuous_rot.pth")
        print(f"  >> Saved best model (AUC={val_auc:.4f})")


  EPOCH 1/6  (LR = 9.33e-04)
  Train                | Loss=0.3551 | Acc=0.8499 | F1=0.8490 | AUC=0.9173
  Validation           | Acc=0.8518 | F1=0.8512 | AUC=0.9221
  ** New best model saved (Val AUC=0.9221) **



  EPOCH 2/6  (LR = 7.50e-04)
  Train                | Loss=0.3132 | Acc=0.8699 | F1=0.8692 | AUC=0.9356
  Validation           | Acc=0.8402 | F1=0.8400 | AUC=0.9169



  EPOCH 3/6  (LR = 5.00e-04)
  Train                | Loss=0.2941 | Acc=0.8791 | F1=0.8785 | AUC=0.9430
  Validation           | Acc=0.8535 | F1=0.8539 | AUC=0.9302
  ** New best model saved (Val AUC=0.9302) **



  EPOCH 4/6  (LR = 2.50e-04)
  Train                | Loss=0.2790 | Acc=0.8854 | F1=0.8848 | AUC=0.9486
  Validation           | Acc=0.8585 | F1=0.8584 | AUC=0.9277

>> Switching to Stage 2: unfreezing layer4, lowering LR




  EPOCH 5/6  (LR = 5.00e-06)
  Train                | Loss=0.2679 | Acc=0.8895 | F1=0.8891 | AUC=0.9525
  Validation           | Acc=0.8594 | F1=0.8589 | AUC=0.9279



  EPOCH 6/6  (LR = 0.00e+00)
  Train                | Loss=0.2667 | Acc=0.8908 | F1=0.8903 | AUC=0.9528
  Validation           | Acc=0.8581 | F1=0.8572 | AUC=0.9290

  FINAL TEST (using best model)



FileNotFoundError: [Errno 2] No such file or directory: 'best_model_continuous_rot_on_training.pth'

In [8]:
print("\nFinal test (best model):")
model.load_state_dict(torch.load("best_model_continuous_rot.pth"))
acc, f1, auc = evaluate(test_loader)
print(f"  Test (0°)  =>  Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")

C:\Users\AhmedFahmy\AppData\Local\Temp\ipykernel_18636\1991932066.py:374: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model_continuo


  FINAL TEST (using best model)



C:\Users\AhmedFahmy\AppData\Local\Temp\ipykernel_18636\408737505.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model_continuous_

  Test (0°)            | Acc=0.8490 | F1=0.8495 | AUC=0.9279

  Experiment complete!
